# Deep Hedging of Knock-Out Barrier Put (100 Steps) (TensorFlow Backend)
This notebook trains neural network deep hedging agents for an up-and-out barrier put option ($K = 1.0, B = 0.95$) over 100 daily time steps ($dt = 0.004$) using TensorFlow/Keras.
Evaluates 2 configurations:
1. `barrier_none`: Non-recurrent feedforward network
2. `barrier_digital`: Recurrent network with digital memory states (past_repr=5, event=5)

**Backend**: TensorFlow 2 / Keras.
**Self-contained**: Uses local `dh_tf` module.


In [ ]:
import os, sys
from pathlib import Path

NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf
from cdxbasics.config import Config
from IPython.display import display, Markdown

from dh_tf.world import SimpleWorld_Spot_ATM
from dh_tf.gym import VanillaDeepHedgingGym
from dh_tf.trainer import train
from configs.catalog import get_config

MODE = os.environ.get("DH_MODE", "original")
def get_epochs(default_ep):
    if os.environ.get("DH_EPOCHS"):
        return int(os.environ.get("DH_EPOCHS"))
    return 2 if MODE == "smoke" else default_ep

def get_samples(default_samples):
    return min(default_samples, 500) if MODE == "smoke" else default_samples

print(f"trainer-recurrent-barrier.ipynb (TF) initialized | Mode: {MODE}")


## Barrier Payoff and World Setup
Knock-out Put ($K = 1.0, B = 0.95$) with causal indicator `np.cumprod`.


In [ ]:
strike = 1.0
barrier = 0.95
def payoff_def(spots):
    notko = np.cumprod(np.where(spots > barrier, 1.0, 0.0), axis=1)
    terminal = np.maximum(strike - spots[:, -1], 0.0)
    return dict(payoff=-terminal * notko[:, -1], features=notko[:, :-1])

payoff_def.name = f"KO Put k={strike}, b={barrier}"

cfg_ref = get_config("barrier_none", mode=MODE)
n_samples = get_samples(cfg_ref["samples"])
n_epochs = get_epochs(cfg_ref["epochs"])
batch_size = min(cfg_ref["batch_size"], n_samples)

config = Config()
config.world.samples = n_samples
config.world.steps = cfg_ref["steps"]
config.world.dt = cfg_ref["dt"]
config.world.drift = cfg_ref["drift"]
config.world.cost_s = cfg_ref["cost_s"]
config.world.black_scholes = True
config.world.payoff = payoff_def

config.gym.objective.utility = cfg_ref["utility"]
config.gym.objective.lmbda = cfg_ref["lmbda"]
config.gym.agent.network.depth = cfg_ref["depth"]
config.gym.agent.network.width = cfg_ref["width"]
config.gym.agent.network.activation = cfg_ref["activation"]
config.gym.agent.features = ['price', 'delta', 'time_left', 'payoff_features']

config.trainer.train.optimizer.name = "adam"
config.trainer.train.optimizer.learning_rate = cfg_ref["lr"]
config.trainer.train.optimizer.clipvalue = cfg_ref["clipvalue"]
config.trainer.train.optimizer.global_clipnorm = cfg_ref["global_clipnorm"]
config.trainer.train.batch_size = batch_size
config.trainer.train.epochs = n_epochs
config.trainer.caching.mode = "off"
config.trainer.output_level = "all" if MODE == "original" else "quiet"

print(f"Simulating Barrier World: {n_samples} samples, {cfg_ref['steps']} steps...")
world = SimpleWorld_Spot_ATM(config.world)
val_world = world.clone(samples=max(n_samples // 2, 50))


## 1. `barrier_none`: Non-Recurrent Training


In [ ]:
config_none = config.copy()
config_none.gym.agent.recurrence.states.classic = 0
config_none.gym.agent.recurrence.states.aggregate = 0
config_none.gym.agent.recurrence.states.past_repr = 0
config_none.gym.agent.recurrence.states.event = 0
config_none.world.mark_done()

print(f"Training barrier_none for {n_epochs} epochs...")
gym_none = VanillaDeepHedgingGym(config_none.gym)
train(gym=gym_none, world=world, val_world=val_world, config=config_none.trainer)
print("Finished barrier_none.")


## 2. `barrier_digital`: Digital Memory Recurrence Training


In [ ]:
config_digital = config.copy()
config_digital.gym.agent.recurrence.states.classic = 0
config_digital.gym.agent.recurrence.states.aggregate = 0
config_digital.gym.agent.recurrence.states.past_repr = 5
config_digital.gym.agent.recurrence.states.event = 5
config_digital.world.mark_done()

print(f"Training barrier_digital for {n_epochs} epochs...")
gym_digital = VanillaDeepHedgingGym(config_digital.gym)
train(gym=gym_digital, world=world, val_world=val_world, config=config_digital.trainer)
print("Finished barrier_digital.")


### Summary
Completed training for both barrier options under TensorFlow backend.
